In [ ]:
!pip install scikit-learn
!pip install pandas
!pip install sentencepiece

In [1]:
# filter dataset
!python3 MT-Preparation/filtering/filter.py ./en-zh.en ./en-zh.zh en zh

Dataframe shape (rows, columns): (231267, 2)
--- Rows with Empty Cells Deleted	--> Rows: 231267
--- Duplicates Deleted			--> Rows: 229646
--- Source-Copied Rows Deleted		--> Rows: 229640
--- Too Long Source/Target Deleted	--> Rows: 224743
--- HTML Removed			--> Rows: 224743
--- Rows will remain true-cased		--> Rows: 224743
--- Rows with Empty Cells Deleted	--> Rows: 224743
--- Source Saved: ./en-zh.en-filtered.en
--- Target Saved: ./en-zh.zh-filtered.zh


In [21]:
import numpy as np
import yake
import multiprocessing as mp
from tqdm import tqdm


def extract_yake_chunk(chunk_indices, corpus, window_size, top_k, worker_id=None):
    kw_extractor = yake.KeywordExtractor(lan="en", n=1, top=top_k)
    chunk_result = []

    for i in chunk_indices:
        if i < window_size // 2:
            num_before = i
            num_after = window_size - num_before
        elif i > len(corpus) - window_size // 2 - 1:
            num_after = len(corpus) - i - 1
            num_before = window_size - num_after
        else:
            num_before = window_size // 2
            num_after = window_size // 2

        before = list(range(max(0, i - num_before), i))
        after = list(range(i + 1, min(len(corpus), i + 1 + num_after)))
        context_indices = before + after
        pseudo_doc = [corpus[j] for j in context_indices]

        if not pseudo_doc:
            chunk_result.append((i, []))
            continue

        combined_text = " ".join(pseudo_doc)
        keywords = kw_extractor.extract_keywords(combined_text)
        sorted_keywords = sorted(keywords, key=lambda x: x[1])
        salient_words = [kw for kw, _ in sorted_keywords[:top_k]]
        chunk_result.append((i, salient_words))

    print(f"[Worker {worker_id}] Finished processing {len(chunk_indices)} lines.")
    return chunk_result


def split_indices_evenly(total, num_chunks):
    chunk_size = (total + num_chunks - 1) // num_chunks
    return [list(range(i * chunk_size, min((i + 1) * chunk_size, total))) for i in range(num_chunks)]


def extract_salient_parallel_chunked(corpus, window_size=4, top_k=5, num_workers=4):
    index_chunks = split_indices_evenly(len(corpus), num_workers)
    args = [(chunk, corpus, window_size, top_k, wid) for wid, chunk in enumerate(index_chunks)]

    with mp.Pool(num_workers) as pool:
        results = pool.starmap(extract_yake_chunk, args)

    # Flatten and reorder results
    flattened = [item for chunk in results for item in chunk]
    sorted_results = [salient for _, salient in sorted(flattened, key=lambda x: x[0])]
    return sorted_results


def write_salient_prefixed_raw_file(raw_lines, output_file, salient_word_lists):
    with open(output_file, "w") as f:
        for line, salient_words in zip(raw_lines, salient_word_lists):
            prefixed_line = ' '.join(salient_words + ['__SEP__'] + line.split())
            f.write(prefixed_line + '\n')
    print(f"[INFO] Wrote output to {output_file}")


# Run
if __name__ == "__main__":
    print("[INFO] Loading input...")
    raw_lines = open("en-zh.en-filtered.en", "r").read().splitlines()
    print(f"[INFO] Loaded {len(raw_lines)} lines.")

    salient_contexts = extract_salient_parallel_chunked(
        raw_lines,
        window_size=4,
        top_k=3,
        num_workers=mp.cpu_count()
    )

    write_salient_prefixed_raw_file(raw_lines, "en-zh.en-filtered-salient.en", salient_contexts)


[INFO] Loading input...
[INFO] Loaded 224743 lines.
[Worker 9] Finished processing 3512 lines.
[Worker 18] Finished processing 3512 lines.
[Worker 1] Finished processing 3512 lines.
[Worker 21] Finished processing 3512 lines.
[Worker 10] Finished processing 3512 lines.
[Worker 17] Finished processing 3512 lines.
[Worker 16] Finished processing 3512 lines.
[Worker 3] Finished processing 3512 lines.
[Worker 0] Finished processing 3512 lines.
[Worker 13] Finished processing 3512 lines.
[Worker 20] Finished processing 3512 lines.
[Worker 8] Finished processing 3512 lines.
[Worker 5] Finished processing 3512 lines.
[Worker 11] Finished processing 3512 lines.
[Worker 28] Finished processing 3512 lines.
[Worker 12] Finished processing 3512 lines.
[Worker 32] Finished processing 3512 lines.
[Worker 2] Finished processing 3512 lines.
[Worker 15] Finished processing 3512 lines.
[Worker 19] Finished processing 3512 lines.
[Worker 4] Finished processing 3512 lines.
[Worker 25] Finished processing 

In [ ]:
salient_words_per_sentence = extract_balanced_salient_words_tfidf(raw_lines)

In [14]:
# train a sentencepiece model for subwording
!python3 MT-Preparation/subwording/1-train_unigram.py ./en-zh.en-filtered-salient.en ./en-zh.zh-filtered.zh

sentencepiece_trainer.cc(178) LOG(INFO) Running command: --input=./en-zh.en-filtered-salient.en --model_prefix=source --vocab_size=10000 --hard_vocab_limit=false --split_digits=true --user_defined_symbols=__SEP__
sentencepiece_trainer.cc(78) LOG(INFO) Starts training with : 
trainer_spec {
  input: ./en-zh.en-filtered-salient.en
  input_format: 
  model_prefix: source
  model_type: UNIGRAM
  vocab_size: 10000
  self_test_sample_size: 0
  character_coverage: 0.9995
  input_sentence_size: 0
  shuffle_input_sentence: 1
  seed_sentencepiece_size: 1000000
  shrinking_factor: 0.75
  max_sentence_length: 4192
  num_threads: 16
  num_sub_iterations: 2
  max_sentencepiece_length: 16
  split_by_unicode_script: 1
  split_by_number: 1
  split_by_whitespace: 1
  split_digits: 1
  pretokenization_delimiter: 
  treat_whitespace_as_suffix: 0
  allow_whitespace_only_pieces: 0
  user_defined_symbols: __SEP__
  required_chars: 
  byte_fallback: 0
  vocabulary_output_piece_score: 1
  train_extremely_large

In [15]:
# subword the dataset
!python3 MT-Preparation/subwording/2-subword.py source.model target.model ./en-zh.en-filtered-salient.en ./en-zh.zh-filtered.zh

Source Model: source.model
Target Model: target.model
Source Dataset: ./en-zh.en-filtered-salient.en
Target Dataset: ./en-zh.zh-filtered.zh
Done subwording the source file! Output: ./en-zh.en-filtered-salient.en.subword
Done subwording the target file! Output: ./en-zh.zh-filtered.zh.subword


In [16]:
# first 3 lines before subwording
!head -n 3 ./en-zh.en-filtered-salient.en && echo "-----" && head -n 3 ./en-zh.zh-filtered.zh

Chris grateful great honor __SEP__ en
conference night blown nice __SEP__ Thank you so much, Chris. And it's truly a great honor to have the opportunity to come to this stage twice; I'm extremely grateful.
Chris grateful Air Force __SEP__ I have been blown away by this conference, and I want to thank all of you for the many nice comments about what I had to say the other night.
-----
zh
非常谢谢，克里斯。的确非常荣幸 能有第二次站在这个台上的机会，我真是非常感激。
这个会议真是让我感到惊叹不已，我还要谢谢你们留下的 关于我上次演讲的精彩评论


In [17]:
# first 3 lines after subwording
!head -n 3 ./en-zh.en-filtered-salient.en.subword && echo "---" && head -n 3 ./en-zh.zh-filtered.zh.subword after

▁Chris ▁grateful ▁great ▁honor ▁ __SEP__ ▁ en
▁conference ▁night ▁ blown ▁nice ▁ __SEP__ ▁Thank ▁you ▁so ▁much , ▁Chris . ▁And ▁it ' s ▁truly ▁a ▁great ▁honor ▁to ▁have ▁the ▁opportunity ▁to ▁come ▁to ▁this ▁stage ▁twice ; ▁I ' m ▁extremely ▁grateful .
▁Chris ▁grateful ▁Air ▁Force ▁ __SEP__ ▁I ▁have ▁been ▁ blown ▁away ▁by ▁this ▁conference , ▁and ▁I ▁want ▁to ▁thank ▁all ▁of ▁you ▁for ▁the ▁many ▁nice ▁comment s ▁about ▁what ▁I ▁had ▁to ▁say ▁the ▁other ▁night .
---
==> ./en-zh.zh-filtered.zh.subword <==
▁ z h
▁非常 谢谢 , 克里斯 。 的确 非常 荣幸 ▁能 有 第二次 站在 这个 台上 的机会 , 我 真是 非常 感激 。
▁这个 会议 真是 让我 感到 惊 叹 不 已 , 我 还要 谢谢你们 留下 的 ▁关于 我 上 次 演讲 的 精彩 评论
head: cannot open 'after' for reading: No such file or directory


In [18]:
# split the dataset into training set, development set, and test set
# Development and test sets should be between 1000 and 5000 segments (here we chose 200)
!python3 MT-Preparation/train_dev_split/train_dev_test_split.py 2000 2000 ./en-zh.en-filtered-salient.en.subword ./en-zh.zh-filtered.zh.subword

Dataframe shape: (224743, 2)
--- Empty Cells Deleted --> Rows: 224743
--- Wrote Files
Done!
Output files
./en-zh.en-filtered-salient.en.subword.train
./en-zh.zh-filtered.zh.subword.train
./en-zh.en-filtered-salient.en.subword.dev
./en-zh.zh-filtered.zh.subword.dev
./en-zh.en-filtered-salient.en.subword.test
./en-zh.zh-filtered.zh.subword.test


In [19]:
!wc -l ./*.subword.*

    2000 ./en-zh.en-filtered-salient.en.subword.dev
    2000 ./en-zh.en-filtered-salient.en.subword.test
    2000 ./en-zh.en-filtered-salient.en.subword.test.desubword
  220743 ./en-zh.en-filtered-salient.en.subword.train
    2000 ./en-zh.zh-filtered.zh.subword.dev
    2000 ./en-zh.zh-filtered.zh.subword.test
    1999 ./en-zh.zh-filtered.zh.subword.test.cleaned
    2000 ./en-zh.zh-filtered.zh.subword.test.desubword
  220743 ./en-zh.zh-filtered.zh.subword.train
  455485 total


In [20]:
# check the first and last line from each dataset
!echo "---First line---"
!head -n 1 ./*.{train,dev,test}

!echo -e "\n---Last line---"
!tail -n 1 ./*.{train,dev,test}

---First line---
==> ./en-zh.en-filtered-salient.en.subword.train <==
▁Chris ▁grateful ▁great ▁honor ▁ __SEP__ ▁ en

==> ./en-zh.zh-filtered.zh.subword.train <==
▁ z h

==> ./en-zh.en-filtered-salient.en.subword.dev <==
▁States ▁rural ▁Ma ine ▁Obama ▁ __SEP__ ▁So ▁it ' s ▁the ▁combination ▁of ▁these ▁two ▁things : ▁it ' s ▁education ▁and ▁the ▁type ▁of ▁neighbors ▁that ▁you ▁have , ▁which ▁we ' ll ▁talk ▁about ▁more ▁in ▁a ▁moment .

==> ./en-zh.zh-filtered.zh.subword.dev <==
▁所以 这两 样东西 是 联合 起来 的 。 ▁其实 就是 你的 受 教育 程度 和 周围 邻居 的 类型 , ▁我们 一会儿 再 具体 的 谈 一 谈 。

==> ./en-zh.en-filtered-salient.en.subword.test <==
▁years ▁Boston ▁Gene ▁Sha r p ▁ __SEP__ ▁D ic t ator ship s ▁in ▁C z e ch os lo v ak ia , ▁East ▁Germany , ▁Estonia , ▁La t vi a , ▁Li t hu an ia , ▁Ma li , ▁Madagascar , ▁Poland , ▁the ▁Philippines , ▁Serbia , ▁S lo ve n ia , ▁I ▁could ▁go ▁on , ▁and ▁now ▁Tunisia ▁and ▁Egypt .

==> ./en-zh.zh-filtered.zh.subword.test <==
▁在 捷 克斯 洛 伐 克 , 东 德 ▁ 爱 沙 尼亚 , 拉 脱 维 亚 , 立 陶 宛 , ▁ 马 里 , 马 达 加